# PPM Benchmark Library — Metric View (Semantic Layer)

This notebook creates a **Unity Catalog Metric View** for the PPM Benchmark Library Genie space.  
A metric view is a centralized definition of all your KPIs/metrics + the dimensions (ways to slice them), stored as a single reusable object in Unity Catalog.

## What's Inside

| Section | What It Does (Layman's Terms) |
| --- | --- |
| **Source** | Points to the raw data table (`ppm_benchmark`) |
| **Filter** | Excludes "Library" placeholder records so only real customer data is analyzed |
| **Dimensions** | The "slicers" — ways to break down metrics (by Customer, Payer, LOB, DP, Medical Policy, etc.) |
| **Measures** | The actual KPIs/metrics — GPV%, NPV%, APV%, line counts, dollar values, etc. |
| **Synonyms** | Alternative names users might type (e.g., "LOB" instead of "Line of Business") so Genie finds the right column |
| **Format** | How numbers display (currency with $ signs, percentages with % signs, plain numbers with commas) |
| **Comments** | Plain-English descriptions of what each metric/dimension means |

## How to Use
1. Run the SQL cell below to create/update the metric view in Unity Catalog
2. Add the metric view to your Genie space under **Configure > Data**
3. Genie will use these definitions to generate correct SQL at any dimensional level

In [0]:
%sql
CREATE OR REPLACE VIEW oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark_metrics
WITH METRICS
LANGUAGE YAML
AS $$
version: 1.1
comment: >
  PPM Benchmark Library — Complete Semantic Layer.
  
  This metric view is the single source of truth for all PPM benchmark KPIs.
  It defines every metric at every dimensional level with LOD-safe handling
  built in. When integrated into a Genie space, it enables correct answers
  to any question regardless of which dimension the user slices by.
  
  ARCHITECTURE:
  - Source: Pre-processed SQL that de-duplicates `paid` at its true grain
    (customer × payer_short × final_state) before computing LOD denominators.
  - Level 1 Dimensions: Customer, LOB, Payer, Final State — standard SUM ratios work.
  - Level 2 Dimensions: DP, Medical Policy, Topic — require LOD-safe denominators.
  - Every percentage metric has BOTH a standard and LOD-safe version.
  - Genie should use LOD-safe versions whenever Level 2 dimensions appear in the query.

# ================================================================
# SOURCE — Pre-processed with LOD-safe paid denominators
# ================================================================
# The source de-duplicates paid at (customer, payer_short, final_state) grain
# using a subquery, then joins back to give each row the correct LOD denominator.
# This ensures SUM(paid_lod) never overcounts regardless of grouping level.
# ================================================================
source: >
  SELECT
    t.*,
    pu.paid_at_cust_payer_state AS paid_lod
  FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark t
  INNER JOIN (
    SELECT
      customer,
      payer_short,
      final_state,
      SUM(paid) AS paid_at_cust_payer_state
    FROM (
      SELECT DISTINCT customer, payer_short, final_state, paid
      FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark
      WHERE customer IS NOT NULL AND customer != 'Lib'
    )
    GROUP BY customer, payer_short, final_state
  ) pu
    ON t.customer = pu.customer
    AND t.payer_short = pu.payer_short
    AND t.final_state = pu.final_state
  WHERE t.customer IS NOT NULL AND t.customer != 'Lib'


# ================================================================
# DIMENSIONS — Complete dimensional hierarchy
# ================================================================
# Level 1 ("Additive"): Paid is unique at these levels.
#   → Standard ratio measures (GPV%, NPV%) give correct results.
# Level 2 ("Fan-out"): Same paid repeats across multiple DPs/policies/topics.
#   → LOD-safe ratio measures MUST be used for correct percentages.
# ================================================================
dimensions:

  # ============================================================
  # LEVEL 1: PRIMARY BUSINESS DIMENSIONS
  # Standard ratios are correct when grouping by these.
  # ============================================================

  - name: Customer
    expr: customer
    display_name: Customer
    comment: >
      Health plan customer (payer organization) served by Cotiviti.
      Excludes 'Lib' placeholder. LEVEL 1 — standard ratios correct.
    synonyms:
      - client
      - health plan
      - account
      - customer name

  - name: Customer Segment
    expr: customer_segment
    display_name: Customer Segment
    comment: >
      Segment classification (e.g., size tier, strategic grouping). LEVEL 1.
    synonyms:
      - segment
      - customer tier
      - account segment

  - name: Line of Business
    expr: insurance_desc
    display_name: Line of Business
    comment: >
      Insurance line of business (Commercial, Medicare, Medicaid, FEP).
      Primary coverage category for claims. LEVEL 1 — standard ratios correct.
    synonyms:
      - LOB
      - insurance type
      - coverage type
      - insurance
      - insurance description

  - name: Product Type
    expr: product_type
    display_name: Product Type
    comment: >
      Cotiviti product — ICM (Inpatient) or ICMO (Outpatient). LEVEL 1.
    synonyms:
      - product
      - solution type
      - ICM type

  - name: Claim Type
    expr: claim_type_desc
    display_name: Claim Type
    comment: >
      Type of healthcare claim (Inpatient, Outpatient, Professional). LEVEL 1.
    synonyms:
      - claim category
      - claim type description

  - name: Claim Type OI
    expr: claim_type_desc_oi
    display_name: Claim Type (OI)
    comment: >
      OI-specific claim type classification. LEVEL 1.
    synonyms:
      - OI claim type

  # ============================================================
  # LEVEL 1: PAYER DIMENSIONS
  # These define the grain where paid is unique.
  # ============================================================

  - name: Payer
    expr: payer_short
    display_name: Payer
    comment: >
      Short name of the payer (plan) within a customer.
      Defines the grain where paid is unique. LEVEL 1.
    synonyms:
      - payer short
      - payer name short
      - plan
      - insurance company

  - name: Payer Full Name
    expr: payer_name
    display_name: Payer Full Name
    comment: >
      Full legal/business name of the payer. LEVEL 1.
    synonyms:
      - payer long name
      - full payer

  - name: Super Payer
    expr: super_payer_short
    display_name: Super Payer
    comment: >
      Parent-level payer grouping. Multiple payers roll up. LEVEL 1.
    synonyms:
      - parent payer
      - payer group
      - super payer short

  - name: Super Payer Full Name
    expr: super_payer_name
    display_name: Super Payer Full Name
    comment: >
      Full name of the super payer group. LEVEL 1.
    synonyms:
      - parent payer name
      - super payer name

  - name: Payer State
    expr: payer_state
    display_name: Payer State
    comment: >
      U.S. state where payer is domiciled. LEVEL 1.
    synonyms:
      - state
      - payer location
      - geography

  - name: Discrete State
    expr: discrete_state
    display_name: Discrete State
    comment: >
      Discrete state categorization for geographical analysis. LEVEL 1.
    synonyms:
      - geography discrete
      - state discrete

  - name: Blues Indicator
    expr: blues_indicator
    display_name: Blues Indicator
    comment: >
      Whether payer is Blue Cross Blue Shield (Y/N). LEVEL 1.
    synonyms:
      - is blues
      - BCBS
      - blue cross

  - name: Blues Home Host
    expr: blues_home_host
    display_name: Blues Home/Host
    comment: >
      For Blues plans, Home vs Host processing. LEVEL 1.
    synonyms:
      - home host
      - blues type

  - name: IDN Indicator
    expr: idn_yn
    display_name: IDN Indicator
    comment: >
      Whether customer is an Integrated Delivery Network (Y/N). LEVEL 1.
    synonyms:
      - is IDN
      - integrated delivery network
      - IDN

  # ============================================================
  # LEVEL 1: STATUS / CONFIGURATION DIMENSIONS
  # ============================================================

  - name: Final State
    expr: final_state
    display_name: Final State
    comment: >
      Final disposition of an edit (Deny, Reduce, Allow).
      Defines the grain where paid is unique. LEVEL 1.
    synonyms:
      - disposition
      - edit outcome
      - action taken

  - name: Status
    expr: status
    display_name: Status
    comment: >
      Operational status of the rule/record. LEVEL 1.
    synonyms:
      - rule status
      - active status

  - name: Disabled
    expr: disabled
    display_name: Disabled
    comment: >
      Whether rule is disabled for this customer/payer.
    synonyms:
      - is disabled
      - rule disabled

  - name: Deactivated
    expr: deactivated
    display_name: Deactivated
    comment: >
      Whether rule has been deactivated.
    synonyms:
      - is deactivated
      - rule deactivated

  - name: Flag
    expr: flag
    display_name: Flag
    comment: >
      General-purpose flag for categorization.
    synonyms:
      - indicator
      - rule flag

  - name: Solution
    expr: solution
    display_name: Solution
    comment: >
      Cotiviti solution/product line.
    synonyms:
      - product line
      - solution name

  - name: Policy Bundle Indicator
    expr: policy_bundle_indicator
    display_name: Policy Bundle
    comment: >
      Whether DP belongs to a policy bundle (grouped policies sold together).
    synonyms:
      - bundle
      - is bundled
      - policy bundle

  - name: Policy Package Indicator
    expr: policy_package_indicator
    display_name: Policy Package
    comment: >
      Whether DP is part of a policy package.
    synonyms:
      - package
      - is packaged

  - name: Policy Package Individual
    expr: policy_package_individual_indicator
    display_name: Policy Package Individual
    comment: >
      Whether DP is an individually-sold policy within a package.
    synonyms:
      - individual package
      - standalone policy

  - name: Library Status
    expr: library_status_key
    display_name: Library Status
    comment: >
      Library status classification for this rule.
    synonyms:
      - lib status
      - library status key

  # ============================================================
  # LEVEL 2: RULE / POLICY DIMENSIONS (Fan-out)
  # MUST use LOD-safe ratio measures at these levels.
  # ============================================================

  - name: Decision Point
    expr: dp_key
    display_name: Decision Point (DP)
    comment: >
      Unique identifier for a decision point (editing rule).
      The fundamental unit of payment policy logic.
      LEVEL 2 — MUST use LOD-safe measures for percentages.
    synonyms:
      - DP
      - DP key
      - rule
      - edit rule
      - decision point key

  - name: Decision Point Description
    expr: dp_desc
    display_name: Decision Point Description
    comment: >
      Human-readable description of what the DP checks. LEVEL 2.
    synonyms:
      - DP description
      - rule description
      - DP name

  - name: Medical Policy
    expr: medical_policy
    display_name: Medical Policy
    comment: >
      Medical policy governing related DPs. Groups DPs under
      a clinical/business theme.
      LEVEL 2 — MUST use LOD-safe measures for percentages.
    synonyms:
      - policy
      - med policy
      - clinical policy
      - policy name

  - name: Latest Medical Policy
    expr: latest_medical1_policy
    display_name: Latest Medical Policy
    comment: >
      Most recent medical policy assignment. LEVEL 2.
    synonyms:
      - current policy
      - latest policy

  - name: Topic
    expr: topic
    display_name: Topic
    comment: >
      High-level clinical/business topic grouping
      (Cardiology, Oncology, Modifiers, etc.).
      LEVEL 2 — MUST use LOD-safe measures for percentages.
    synonyms:
      - clinical topic
      - category
      - subject area

  - name: Sub Rule Key
    expr: sub_rule_key
    display_name: Sub Rule Key
    comment: >
      Sub-rule identifier within a DP. More granular than DP. LEVEL 2.
    synonyms:
      - sub rule
      - subrule

  - name: Sub Rule Description
    expr: sub_rule_desc
    display_name: Sub Rule Description
    comment: >
      Human-readable description of the sub-rule. LEVEL 2.
    synonyms:
      - subrule description
      - sub rule name

  - name: Mid Rule Key
    expr: mid_rule_key
    display_name: Mid Rule Key
    comment: >
      Mid-level rule. Hierarchy: DP > Mid Rule > Sub Rule. LEVEL 2.
    synonyms:
      - mid rule
      - MR key

  - name: Mid Rule Version
    expr: mid_rule_version
    display_name: Mid Rule Version
    comment: >
      Version of the mid-level rule.
    synonyms:
      - rule version
      - MR version

  - name: Mid Rule Dot Version
    expr: mid_rule_dot_version
    display_name: Mid Rule Dot Version
    comment: >
      Dot-version (patch level) of the mid-level rule.
    synonyms:
      - dot version
      - patch version

  - name: DP for Sub Rule
    expr: dp_key_4_sub_rule_key
    display_name: DP for Sub Rule
    comment: >
      Parent DP key that owns this sub-rule. Maps sub-rules to parent.
    synonyms:
      - parent DP
      - sub rule parent

  - name: DP Description for Sub Rule
    expr: dp_desc_4_sub_rule_key
    display_name: DP Description for Sub Rule
    comment: >
      Description of the parent DP for this sub-rule.
    synonyms:
      - sub rule parent description

  - name: Topic Key for Sub Rule
    expr: topic_key_4_sub_rule_key
    display_name: Topic Key for Sub Rule
    comment: >
      Topic key associated with this sub-rule.
    synonyms:
      - sub rule topic key

  - name: Topic for Sub Rule
    expr: topic_title_4_sub_rule_key
    display_name: Topic for Sub Rule
    comment: >
      Topic title associated with this sub-rule.
    synonyms:
      - sub rule topic

  - name: Medical Policy Key for Sub Rule
    expr: med_pol_key_4_sub_rule_key
    display_name: Medical Policy Key for Sub Rule
    comment: >
      Medical policy key for this sub-rule.
    synonyms:
      - sub rule policy key

  - name: Medical Policy for Sub Rule
    expr: med_pol_title_4_sub_rule_key
    display_name: Medical Policy for Sub Rule
    comment: >
      Medical policy title for this sub-rule.
    synonyms:
      - sub rule policy

  - name: Policy Type Key for Sub Rule
    expr: pol_type_key_4_sub_rule_key
    display_name: Policy Type Key for Sub Rule
    comment: >
      Policy type key for this sub-rule.
    synonyms:
      - sub rule policy type key

  - name: Policy Type for Sub Rule
    expr: pol_type_desc_4_sub_rule_key
    display_name: Policy Type for Sub Rule
    comment: >
      Policy type description for this sub-rule.
    synonyms:
      - sub rule policy type

  # ============================================================
  # LEVEL 1: OPERATIONAL / TEAM DIMENSIONS
  # ============================================================

  - name: Claims Platform
    expr: claims_platform
    display_name: Claims Platform
    comment: >
      Claims processing platform used by the customer.
    synonyms:
      - platform
      - adjudication system

  - name: Cotiviti Edit Position
    expr: cotiviti_edit_position
    display_name: Cotiviti Edit Position
    comment: >
      Position in processing sequence (pre-pay, post-pay, primary editor).
    synonyms:
      - edit position
      - processing position

  - name: Primary 3rd Party Editor
    expr: primary_3rd_party_editor
    display_name: Primary 3rd Party Editor
    comment: >
      Primary third-party editing vendor alongside Cotiviti.
    synonyms:
      - competitor
      - other editor
      - third party

  - name: Secondary 3rd Party Editor
    expr: secondary_3rd_party_editor
    display_name: Secondary 3rd Party Editor
    comment: >
      Secondary third-party editing vendor.
    synonyms:
      - second competitor
      - other vendor

  - name: SVP
    expr: svp
    display_name: SVP
    comment: >
      Senior Vice President for this customer relationship.
    synonyms:
      - senior VP
      - executive owner

  - name: CEL
    expr: cel
    display_name: CEL
    comment: >
      Client Engagement Lead.
    synonyms:
      - client engagement lead
      - engagement lead

  - name: CSD
    expr: csd
    display_name: CSD
    comment: >
      Client Service Director.
    synonyms:
      - client service director
      - service director

  - name: CSM
    expr: csm
    display_name: CSM
    comment: >
      Client Success Manager.
    synonyms:
      - client success manager
      - success manager

  - name: CPM
    expr: cpm
    display_name: CPM
    comment: >
      Client Program Manager.
    synonyms:
      - client program manager
      - program manager

  - name: BA
    expr: ba
    display_name: BA
    comment: >
      Business Analyst.
    synonyms:
      - business analyst
      - analyst

  - name: Client Medical Director
    expr: client_medical_director
    display_name: Client Medical Director
    comment: >
      Medical Director on the client side.
    synonyms:
      - medical director
      - CMD

  # ============================================================
  # DATE / TIME DIMENSIONS
  # ============================================================

  - name: Max Invoice Year
    expr: max_invoice_year
    display_name: Max Invoice Year
    comment: >
      Most recent invoice year.
    synonyms:
      - invoice year
      - latest year
      - year

  - name: Min Invoice Date
    expr: min_invoice_date
    display_name: Min Invoice Date
    comment: >
      Earliest invoice date.
    synonyms:
      - first invoice
      - start date

  - name: Max Invoice Date
    expr: max_invoice_date
    display_name: Max Invoice Date
    comment: >
      Most recent invoice date.
    synonyms:
      - last invoice
      - end date
      - latest invoice date

  - name: DP Start Date
    expr: dp_start_date
    display_name: DP Start Date
    comment: >
      Date the decision point was first activated.
    synonyms:
      - rule start date
      - deployment date

  - name: Mid Rule Start Date
    expr: mid_rule_start_date
    display_name: Mid Rule Start Date
    comment: >
      Date the mid-rule version was deployed.
    synonyms:
      - MR start date
      - version date

  - name: Latest Ref Date
    expr: latest_ref_date
    display_name: Latest Reference Date
    comment: >
      Latest reference/refresh date for this record.
    synonyms:
      - reference date
      - refresh date

  # ============================================================
  # IDENTIFIER / CODE DIMENSIONS (for joins and lookups)
  # ============================================================

  - name: Insurance Key
    expr: insurance_key
    display_name: Insurance Key
    comment: >
      Surrogate key for the insurance/LOB.
    synonyms:
      - LOB key
      - insurance ID

  - name: Customer Code
    expr: customer_code
    display_name: Customer Code
    comment: >
      Internal code for the customer.
    synonyms:
      - client code
      - account code

  - name: Payer Key
    expr: payer_key
    display_name: Payer Key
    comment: >
      Surrogate key for the payer.
    synonyms:
      - payer ID

  - name: Payer Short Code
    expr: payer_short_code
    display_name: Payer Short Code
    comment: >
      Short code identifier for the payer.
    synonyms:
      - payer code

  - name: Super Payer Key
    expr: super_payer_key
    display_name: Super Payer Key
    comment: >
      Surrogate key for the super payer.
    synonyms:
      - parent payer key

  - name: Super Payer Code
    expr: super_payer_code
    display_name: Super Payer Code
    comment: >
      Code identifier for the super payer.
    synonyms:
      - parent payer code

  # ============================================================
  # OTHER / CLAIMVIEW DIMENSIONS
  # ============================================================

  - name: CV Code
    expr: cv_code
    display_name: CV Code
    comment: >
      ClaimView code classification.
    synonyms:
      - ClaimView code
      - claim view code

  - name: CV Rule 10
    expr: cv_rule_10
    display_name: CV Rule 10
    comment: >
      ClaimView Rule 10 classification.
    synonyms:
      - ClaimView rule

  - name: CV Client
    expr: cv_client
    display_name: CV Client
    comment: >
      ClaimView client identifier.
    synonyms:
      - ClaimView client

  - name: NDC Data
    expr: ndc_data
    display_name: NDC Data
    comment: >
      National Drug Code data indicator.
    synonyms:
      - drug code
      - NDC

  - name: Client Name
    expr: client_name
    display_name: Client Name
    comment: >
      Alternative client/customer name field.
    synonyms:
      - alt customer name

  - name: PCI Master List
    expr: pci_master_list
    display_name: PCI Master List
    comment: >
      PCI master list classification.
    synonyms:
      - PCI list

  - name: PBI Master List
    expr: pbi_master_list
    display_name: PBI Master List
    comment: >
      PBI master list classification.
    synonyms:
      - PBI list

  - name: PCBI Master List
    expr: pcbi_master_list
    display_name: PCBI Master List
    comment: >
      PCBI master list classification.
    synonyms:
      - PCBI list


# ================================================================
# MEASURES — Complete metric definitions at all levels
# ================================================================
# ORGANIZATION:
#   1. CORE DOLLAR METRICS — absolute values, correct at all levels
#   2. STANDARD PERCENTAGE METRICS — correct at Level 1 dimensions
#   3. LOD-SAFE PERCENTAGE METRICS — correct at ALL levels (Level 1 + 2)
#   4. LINE VOLUME METRICS — counts, correct at all levels
#   5. RATE & FREQUENCY METRICS — per-unit calculations
#   6. COUNT / CARDINALITY METRICS — distinct entity counts
#   7. ADOPTION METRICS — penetration and adoption rates
#   8. FILTERED METRICS — conditional measures for specific segments
#   9. DERIVED / COMPOSITE METRICS — built from other measures
# ================================================================
measures:

  # ============================================================
  # 1. CORE DOLLAR METRICS (correct at all levels)
  # These are absolute sums — no denominator, no LOD issue.
  # ============================================================

  - name: Total Paid
    expr: SUM(COALESCE(paid, 0))
    display_name: Total Paid
    comment: >
      Total original paid/allowed amount on claim lines.
      The denominator for most percentage metrics.
      NOTE: At Level 2 dimensions (DP, Policy, Topic) this overcounts
      because paid repeats per DP row. For correct percentages at
      Level 2, use the LOD-safe ratio measures instead.
    format:
      type: currency
      currency_code: USD
      decimal_places:
        type: exact
        places: 0
      abbreviation: compact
    synonyms:
      - paid
      - paid amount
      - total allowed
      - allowed amount
      - dollar volume

  - name: Total GPV
    expr: SUM(COALESCE(gpv, 0))
    display_name: Total GPV
    comment: >
      Gross Payment Value — raw savings identified by edits
      before adjustments. What Cotiviti rules flagged as
      potential overpayments. Correct at all levels.
    format:
      type: currency
      currency_code: USD
      decimal_places:
        type: exact
        places: 0
      abbreviation: compact
    synonyms:
      - GPV
      - gross payment value
      - gross savings
      - identified savings

  - name: Total Adjustments
    expr: SUM(COALESCE(adjustments, 0))
    display_name: Total Adjustments
    comment: >
      Adjustments applied to GPV (disputes, clinical overrides).
      Typically negative — reduces GPV to NPV. Correct at all levels.
    format:
      type: currency
      currency_code: USD
      decimal_places:
        type: exact
        places: 0
      abbreviation: compact
    synonyms:
      - adjustments
      - adjustment amount
      - adjustment value
      - disputes

  - name: Total NPV
    expr: SUM(COALESCE(npv, 0))
    display_name: Total NPV
    comment: >
      Net Payment Value — final confirmed savings after adjustments.
      NPV = GPV + Adjustments. The "real" savings the customer keeps.
      Correct at all levels.
    format:
      type: currency
      currency_code: USD
      decimal_places:
        type: exact
        places: 0
      abbreviation: compact
    synonyms:
      - NPV
      - net payment value
      - net savings
      - actual savings
      - realized savings

  - name: Total EPV
    expr: SUM(COALESCE(epv, 0))
    display_name: Total EPV
    comment: >
      Edit Payment Value — value directly associated with the edit action.
      Correct at all levels.
    format:
      type: currency
      currency_code: USD
      decimal_places:
        type: exact
        places: 0
      abbreviation: compact
    synonyms:
      - EPV
      - edit payment value
      - edit value

  - name: Total Unapplied
    expr: SUM(COALESCE(unapplied, 0))
    display_name: Total Unapplied
    comment: >
      Unapplied value — edits identified but not applied/accepted
      by the customer. Missed opportunity. Correct at all levels.
    format:
      type: currency
      currency_code: USD
      decimal_places:
        type: exact
        places: 0
      abbreviation: compact
    synonyms:
      - unapplied
      - unapplied savings
      - missed savings
      - opportunity

  - name: Total CV Paid
    expr: SUM(COALESCE(cv_paid, 0))
    display_name: Total CV Paid
    comment: >
      ClaimView-specific paid amount. Correct at all levels.
    format:
      type: currency
      currency_code: USD
      decimal_places:
        type: exact
        places: 0
      abbreviation: compact
    synonyms:
      - ClaimView paid
      - CV paid

  - name: Total Paid LOD
    expr: SUM(COALESCE(paid_lod, 0))
    display_name: Total Paid (De-duplicated)
    comment: >
      Total paid de-duplicated at the (customer, payer, state) grain.
      This is the correct denominator for percentage calculations
      at ALL levels including Level 2 (DP, Policy, Topic).
      When grouping by Level 1 dimensions, this equals Total Paid.
      When grouping by Level 2 dimensions, this prevents overcounting.
    format:
      type: currency
      currency_code: USD
      decimal_places:
        type: exact
        places: 0
      abbreviation: compact
    synonyms:
      - paid deduped
      - true paid
      - LOD paid

  # ============================================================
  # 2. STANDARD PERCENTAGE METRICS
  # Correct ONLY at Level 1 dimensions (Customer, LOB, Payer, State).
  # Use for: Overall, by Customer, by LOB, by Payer, by Final State.
  # DO NOT use at: DP, Medical Policy, Topic levels.
  # ============================================================

  - name: GPV Percent
    expr: 100.0 * SUM(COALESCE(gpv, 0)) / NULLIF(SUM(COALESCE(paid, 0)), 0)
    display_name: GPV%
    comment: >
      Gross Payment Value as % of paid. "Cents saved per dollar (gross)."
      CORRECT AT: Overall, Customer, LOB, Payer, Final State.
      WRONG AT: DP, Medical Policy, Topic (use GPV% LOD-Safe instead).
    format:
      type: number
      decimal_places:
        type: exact
        places: 3
    synonyms:
      - GPV%
      - GPV percent
      - gross savings rate
      - savings percent

  - name: NPV Percent
    expr: 100.0 * SUM(COALESCE(npv, 0)) / NULLIF(SUM(COALESCE(paid, 0)), 0)
    display_name: NPV%
    comment: >
      Net Payment Value as % of paid. "Cents saved per dollar (net)."
      The bottom-line savings rate after adjustments.
      CORRECT AT: Overall, Customer, LOB, Payer, Final State.
      WRONG AT: DP, Medical Policy, Topic (use NPV% LOD-Safe instead).
    format:
      type: number
      decimal_places:
        type: exact
        places: 3
    synonyms:
      - NPV%
      - NPV percent
      - net savings rate
      - actual savings rate

  - name: APV Percent
    expr: 100.0 * SUM(COALESCE(adjustments, 0)) / NULLIF(SUM(COALESCE(gpv, 0)), 0)
    display_name: APV%
    comment: >
      Adjustment % of GPV. "How much got reversed/disputed."
      Typically negative. Closer to 0% = better savings retention.
      NOTE: Since denominator is GPV (not paid), this is actually
      correct at all levels because GPV is unique per DP.
    format:
      type: number
      decimal_places:
        type: exact
        places: 2
    synonyms:
      - APV%
      - APV percent
      - adjustment rate
      - reversal rate

  - name: EPV Percent
    expr: 100.0 * SUM(COALESCE(epv, 0)) / NULLIF(SUM(COALESCE(paid, 0)), 0)
    display_name: EPV%
    comment: >
      Edit Payment Value as % of paid.
      CORRECT AT: Level 1 dimensions only.
    format:
      type: number
      decimal_places:
        type: exact
        places: 3
    synonyms:
      - EPV%
      - EPV percent
      - edit value rate

  - name: Unapplied Percent
    expr: |-
      100.0 * SUM(COALESCE(unapplied, 0)) /
      NULLIF(SUM(COALESCE(unapplied, 0)) + SUM(COALESCE(gpv, 0)), 0)
    display_name: Unapplied%
    comment: >
      % of total potential (unapplied + GPV) that went unapplied.
      "How much is left on the table." Higher = more missed opportunity.
      Since denominator uses GPV (not paid), correct at most levels.
    format:
      type: number
      decimal_places:
        type: exact
        places: 2
    synonyms:
      - unapplied%
      - unapplied percent
      - missed opportunity rate
      - opportunity percent

  # ============================================================
  # 3. LOD-SAFE PERCENTAGE METRICS
  # Correct at ALL dimensional levels including Level 2.
  # Uses pre-computed paid_lod denominator from de-duplicated source.
  # Use these whenever DP, Medical Policy, Topic, or Sub Rule
  # appears in the GROUP BY.
  # ============================================================

  - name: GPV Percent LOD Safe
    expr: |-
      100.0 * SUM(COALESCE(gpv, 0)) /
      NULLIF(SUM(COALESCE(paid_lod, 0)), 0)
    display_name: GPV% (All Levels)
    comment: >
      LOD-safe GPV%. Uses de-duplicated paid denominator.
      CORRECT AT ALL LEVELS — Overall, Customer, LOB, Payer,
      DP Key, Medical Policy, Topic, Sub Rule.
      This is the PRIMARY measure for GPV% when slicing by
      any rule-level dimension. Answers: "What share of total
      paid does this DP/policy/topic account for in gross savings?"
    format:
      type: number
      decimal_places:
        type: exact
        places: 4
    synonyms:
      - GPV% LOD
      - GPV% safe
      - GPV% all levels
      - GPV% by DP
      - GPV% by policy
      - GPV% by topic

  - name: NPV Percent LOD Safe
    expr: |-
      100.0 * SUM(COALESCE(npv, 0)) /
      NULLIF(SUM(COALESCE(paid_lod, 0)), 0)
    display_name: NPV% (All Levels)
    comment: >
      LOD-safe NPV%. Uses de-duplicated paid denominator.
      CORRECT AT ALL LEVELS — Overall, Customer, LOB, Payer,
      DP Key, Medical Policy, Topic, Sub Rule.
      This is the PRIMARY measure for NPV% when slicing by
      any rule-level dimension. Answers: "What share of total
      paid does this DP/policy/topic account for in net savings?"
    format:
      type: number
      decimal_places:
        type: exact
        places: 4
    synonyms:
      - NPV% LOD
      - NPV% safe
      - NPV% all levels
      - NPV% by DP
      - NPV% by policy
      - NPV% by topic

  - name: EPV Percent LOD Safe
    expr: |-
      100.0 * SUM(COALESCE(epv, 0)) /
      NULLIF(SUM(COALESCE(paid_lod, 0)), 0)
    display_name: EPV% (All Levels)
    comment: >
      LOD-safe EPV%. Uses de-duplicated paid denominator.
      CORRECT AT ALL LEVELS.
    format:
      type: number
      decimal_places:
        type: exact
        places: 4
    synonyms:
      - EPV% LOD
      - EPV% safe
      - EPV% all levels

  # ============================================================
  # 4. LINE VOLUME METRICS (correct at all levels)
  # These are simple sums — no denominator issue.
  # ============================================================

  - name: Total Lines
    expr: SUM(COALESCE(total_lines, 0))
    display_name: Total Lines
    comment: >
      Total claim lines processed (universe that could be edited).
      Denominator for edit rate metrics. Correct at all levels.
    format:
      type: number
      decimal_places:
        type: exact
        places: 0
    synonyms:
      - total claim lines
      - TBA lines
      - lines processed
      - volume

  - name: Edited Lines
    expr: SUM(COALESCE(edited_lines, 0))
    display_name: Edited Lines
    comment: >
      Lines where at least one rule fired. Correct at all levels.
    format:
      type: number
      decimal_places:
        type: exact
        places: 0
    synonyms:
      - edits
      - lines edited
      - edited count

  - name: Adjusted Lines
    expr: SUM(COALESCE(adjusted_lines, 0))
    display_name: Adjusted Lines
    comment: >
      Lines where the edit resulted in an actual payment change.
      Subset of edited lines. Correct at all levels.
    format:
      type: number
      decimal_places:
        type: exact
        places: 0
    synonyms:
      - adjusted line count
      - lines adjusted
      - successful edits

  # ============================================================
  # 5. RATE & FREQUENCY METRICS
  # Per-unit, ratio, and frequency calculations.
  # ============================================================

  - name: Adjusted Lines Percent
    expr: |-
      100.0 * SUM(COALESCE(adjusted_lines, 0)) /
      NULLIF(SUM(COALESCE(edited_lines, 0)), 0)
    display_name: Adjusted Lines %
    comment: >
      % of edited lines that resulted in actual adjustments.
      "Edit success rate." Correct at all levels (no paid involved).
    format:
      type: number
      decimal_places:
        type: exact
        places: 2
    synonyms:
      - adjusted lines%
      - adjustment rate lines
      - edit success rate
      - accuracy rate

  - name: Edits per 1000
    expr: |-
      1000.0 * SUM(COALESCE(edited_lines, 0)) /
      NULLIF(SUM(COALESCE(total_lines, 0)), 0)
    display_name: Edits per 1,000
    comment: >
      Edits per 1,000 total lines processed. Normalized edit frequency.
      Correct at all levels (no paid involved).
    format:
      type: number
      decimal_places:
        type: exact
        places: 2
    synonyms:
      - edit rate
      - edits per thousand
      - edit frequency
      - hit rate lines

  - name: Original Paid per Edit
    expr: |-
      SUM(CASE WHEN hit_flag = 1 THEN COALESCE(paid, 0) ELSE 0 END) /
      NULLIF(SUM(CASE WHEN hit_flag = 1 THEN COALESCE(edited_lines, 0) ELSE 0 END), 0)
    display_name: Original Paid per Edit
    comment: >
      Average original paid on lines that were edited (hit).
      "How expensive are flagged claim lines?" Higher = targeting
      higher-value claims. Correct at Level 1.
    format:
      type: currency
      currency_code: USD
      decimal_places:
        type: exact
        places: 0
    synonyms:
      - paid per edit
      - average paid per edit
      - dollars per edit
      - value per edit

  - name: Original Paid per TBA Line
    expr: |-
      SUM(COALESCE(paid, 0)) /
      NULLIF(SUM(COALESCE(total_lines, 0)), 0)
    display_name: Original Paid per TBA Line
    comment: >
      Average paid per total line processed.
      "Average dollar value of each claim line flowing through."
    format:
      type: currency
      currency_code: USD
      decimal_places:
        type: exact
        places: 0
    synonyms:
      - paid per line
      - average paid per line
      - dollars per line
      - average line value

  - name: GPV per Edit
    expr: |-
      SUM(COALESCE(gpv, 0)) /
      NULLIF(SUM(COALESCE(edited_lines, 0)), 0)
    display_name: GPV per Edit
    comment: >
      Average gross savings per edited line.
      "How much does each edit save (gross)?" Correct at all levels.
    format:
      type: currency
      currency_code: USD
      decimal_places:
        type: exact
        places: 2
    synonyms:
      - savings per edit
      - GPV per line
      - gross savings per edit

  - name: NPV per Edit
    expr: |-
      SUM(COALESCE(npv, 0)) /
      NULLIF(SUM(COALESCE(edited_lines, 0)), 0)
    display_name: NPV per Edit
    comment: >
      Average net savings per edited line.
      "How much does each edit actually save (net)?" Correct at all levels.
    format:
      type: currency
      currency_code: USD
      decimal_places:
        type: exact
        places: 2
    synonyms:
      - net savings per edit
      - NPV per line

  - name: Adjustment per GPV Line
    expr: |-
      SUM(COALESCE(adjustments, 0)) /
      NULLIF(SUM(COALESCE(adjusted_lines, 0)), 0)
    display_name: Adjustment per Adjusted Line
    comment: >
      Average adjustment amount per adjusted line.
      Correct at all levels.
    format:
      type: currency
      currency_code: USD
      decimal_places:
        type: exact
        places: 2
    synonyms:
      - adjustment per line
      - average adjustment

  # ============================================================
  # 6. COUNT / CARDINALITY METRICS
  # Distinct counts of entities. Correct at all levels.
  # ============================================================

  - name: Customer Count
    expr: COUNT(DISTINCT customer)
    display_name: Customer Count
    comment: >
      Number of distinct customers. Correct at all levels.
    format:
      type: number
      decimal_places:
        type: exact
        places: 0
    synonyms:
      - number of customers
      - client count
      - account count

  - name: Payer Count
    expr: |-
      COUNT(DISTINCT CASE
        WHEN payer_short IS NOT NULL AND UPPER(TRIM(payer_short)) != 'LIB'
        THEN payer_short
      END)
    display_name: Payer Count
    comment: >
      Number of distinct payers, excluding library placeholders.
      Correct at all levels.
    format:
      type: number
      decimal_places:
        type: exact
        places: 0
    synonyms:
      - number of payers
      - plan count
      - payer volume

  - name: DP Count
    expr: COUNT(DISTINCT dp_key)
    display_name: DP Count
    comment: >
      Number of distinct decision points. Correct at all levels.
    format:
      type: number
      decimal_places:
        type: exact
        places: 0
    synonyms:
      - number of DPs
      - rule count
      - decision point count

  - name: Medical Policy Count
    expr: COUNT(DISTINCT medical_policy)
    display_name: Medical Policy Count
    comment: >
      Number of distinct medical policies. Correct at all levels.
    format:
      type: number
      decimal_places:
        type: exact
        places: 0
    synonyms:
      - number of policies
      - policy count

  - name: Topic Count
    expr: COUNT(DISTINCT topic)
    display_name: Topic Count
    comment: >
      Number of distinct topics. Correct at all levels.
    format:
      type: number
      decimal_places:
        type: exact
        places: 0
    synonyms:
      - number of topics
      - category count

  - name: Sub Rule Count
    expr: COUNT(DISTINCT sub_rule_key)
    display_name: Sub Rule Count
    comment: >
      Number of distinct sub-rules. Correct at all levels.
    format:
      type: number
      decimal_places:
        type: exact
        places: 0
    synonyms:
      - number of sub rules
      - subrule count

  - name: Super Payer Count
    expr: COUNT(DISTINCT super_payer_short)
    display_name: Super Payer Count
    comment: >
      Number of distinct super payer groups. Correct at all levels.
    format:
      type: number
      decimal_places:
        type: exact
        places: 0
    synonyms:
      - number of super payers
      - parent payer count

  # ============================================================
  # 7. ADOPTION METRICS
  # Customer/payer penetration and rule adoption rates.
  # ============================================================

  - name: Customer Adoption Rate
    expr: |-
      100.0 * COUNT(DISTINCT customer) /
      NULLIF((SELECT COUNT(DISTINCT customer)
              FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark
              WHERE customer IS NOT NULL AND customer != 'Lib'), 0)
    display_name: Customer Adoption Rate
    comment: >
      % of ALL customers that have this rule/policy/topic.
      Most meaningful when filtered by a specific DP or Policy.
      "Out of all our customers, what fraction uses this?"
    format:
      type: number
      decimal_places:
        type: exact
        places: 1
    synonyms:
      - adoption rate
      - adoption percent
      - customer penetration
      - client adoption

  - name: Payer Adoption Rate
    expr: |-
      100.0 * COUNT(DISTINCT CASE
        WHEN payer_short IS NOT NULL AND UPPER(TRIM(payer_short)) != 'LIB'
        THEN payer_short END) /
      NULLIF((SELECT COUNT(DISTINCT CASE
        WHEN payer_short IS NOT NULL AND UPPER(TRIM(payer_short)) != 'LIB'
        THEN payer_short END)
              FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark
              WHERE customer IS NOT NULL AND customer != 'Lib'), 0)
    display_name: Payer Adoption Rate
    comment: >
      % of ALL payers that have this rule/policy/topic.
      "Out of all payers, what fraction uses this?"
    format:
      type: number
      decimal_places:
        type: exact
        places: 1
    synonyms:
      - payer penetration
      - plan adoption
      - payer adoption percent

  # ============================================================
  # 8. FILTERED METRICS
  # Conditional measures for specific segments/flags.
  # ============================================================

  - name: Hit Line Count
    expr: SUM(COALESCE(hit_flag, 0))
    display_name: Hit Line Count
    comment: >
      Rows where an edit actually hit (hit_flag = 1).
      Correct at all levels.
    format:
      type: number
      decimal_places:
        type: exact
        places: 0
    synonyms:
      - hits
      - hit count
      - fired count

  - name: CV Flag Count
    expr: SUM(COALESCE(cv_flag, 0))
    display_name: CV Flag Count
    comment: >
      ClaimView-flagged record count. Correct at all levels.
    format:
      type: number
      decimal_places:
        type: exact
        places: 0
    synonyms:
      - ClaimView flags
      - CV flags

  - name: CV OP Active Client Flag
    expr: SUM(COALESCE(cv_op_flag_active_client, 0))
    display_name: CV OP Active Client
    comment: >
      ClaimView outpatient active client flag count.
    format:
      type: number
      decimal_places:
        type: exact
        places: 0
    synonyms:
      - CV OP active
      - ClaimView outpatient active

  - name: CV Prof Active Client Flag
    expr: SUM(COALESCE(cv_prof_flag_active_client, 0))
    display_name: CV Prof Active Client
    comment: >
      ClaimView professional active client flag count.
    format:
      type: number
      decimal_places:
        type: exact
        places: 0
    synonyms:
      - CV Prof active
      - ClaimView professional active

  - name: GPV from Hits Only
    expr: SUM(CASE WHEN hit_flag = 1 THEN COALESCE(gpv, 0) ELSE 0 END)
    display_name: GPV (Hits Only)
    comment: >
      GPV only from lines where the edit hit. Correct at all levels.
    format:
      type: currency
      currency_code: USD
      decimal_places:
        type: exact
        places: 0
      abbreviation: compact
    synonyms:
      - hit GPV
      - GPV hits
      - GPV where hit

  - name: NPV from Hits Only
    expr: SUM(CASE WHEN hit_flag = 1 THEN COALESCE(npv, 0) ELSE 0 END)
    display_name: NPV (Hits Only)
    comment: >
      NPV only from lines where the edit hit. Correct at all levels.
    format:
      type: currency
      currency_code: USD
      decimal_places:
        type: exact
        places: 0
      abbreviation: compact
    synonyms:
      - hit NPV
      - NPV hits
      - NPV where hit

  - name: Paid from Hits Only
    expr: SUM(CASE WHEN hit_flag = 1 THEN COALESCE(paid, 0) ELSE 0 END)
    display_name: Paid (Hits Only)
    comment: >
      Paid amount only from lines where the edit hit.
    format:
      type: currency
      currency_code: USD
      decimal_places:
        type: exact
        places: 0
      abbreviation: compact
    synonyms:
      - hit paid
      - paid where hit

  # ============================================================
  # 9. DERIVED / COMPOSITE METRICS
  # Built from combinations of other values.
  # ============================================================

  - name: Net Retention Rate
    expr: |-
      100.0 * SUM(COALESCE(npv, 0)) /
      NULLIF(SUM(COALESCE(gpv, 0)), 0)
    display_name: Net Retention Rate
    comment: >
      % of gross savings retained after adjustments. NPV/GPV × 100.
      Higher = less disputed. Correct at all levels (GPV denominator).
    format:
      type: number
      decimal_places:
        type: exact
        places: 2
    synonyms:
      - retention rate
      - savings retention
      - NPV to GPV ratio

  - name: Hit Rate
    expr: |-
      100.0 * SUM(COALESCE(hit_flag, 0)) /
      NULLIF(COUNT(*), 0)
    display_name: Hit Rate
    comment: >
      % of all rows where an edit hit. Correct at all levels.
    format:
      type: number
      decimal_places:
        type: exact
        places: 2
    synonyms:
      - hit percentage
      - fire rate
      - rule fire rate

  - name: Unapplied Opportunity Value
    expr: |-
      SUM(COALESCE(unapplied, 0)) *
      (SUM(COALESCE(npv, 0)) / NULLIF(SUM(COALESCE(gpv, 0)), 0))
    display_name: Unapplied Opportunity (Net)
    comment: >
      Estimated net value of unapplied savings if they were applied,
      adjusted by the current net retention rate.
      "If we turned on these edits, approximately how much would we save (net)?"
    format:
      type: currency
      currency_code: USD
      decimal_places:
        type: exact
        places: 0
      abbreviation: compact
    synonyms:
      - potential net savings
      - unapplied opportunity
      - missed NPV

  - name: GPV Concentration
    expr: |-
      100.0 * SUM(COALESCE(gpv, 0)) /
      NULLIF((SELECT SUM(COALESCE(gpv, 0))
              FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark
              WHERE customer IS NOT NULL AND customer != 'Lib'), 0)
    display_name: GPV Concentration %
    comment: >
      What share of ALL GPV comes from this slice.
      "This customer/DP/policy accounts for X% of total savings."
      Correct at all levels.
    format:
      type: number
      decimal_places:
        type: exact
        places: 2
    synonyms:
      - GPV share
      - savings share
      - GPV contribution
      - GPV concentration

  - name: NPV Concentration
    expr: |-
      100.0 * SUM(COALESCE(npv, 0)) /
      NULLIF((SELECT SUM(COALESCE(npv, 0))
              FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark
              WHERE customer IS NOT NULL AND customer != 'Lib'), 0)
    display_name: NPV Concentration %
    comment: >
      What share of ALL NPV comes from this slice.
      Correct at all levels.
    format:
      type: number
      decimal_places:
        type: exact
        places: 2
    synonyms:
      - NPV share
      - net savings share
      - NPV contribution
      - NPV concentration

  - name: Paid Concentration
    expr: |-
      100.0 * SUM(COALESCE(paid, 0)) /
      NULLIF((SELECT SUM(COALESCE(paid, 0))
              FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark
              WHERE customer IS NOT NULL AND customer != 'Lib'), 0)
    display_name: Paid Concentration %
    comment: >
      What share of ALL paid comes from this slice.
      Correct at Level 1 dimensions.
    format:
      type: number
      decimal_places:
        type: exact
        places: 2
    synonyms:
      - paid share
      - volume share
      - paid contribution

$$

## SQL Expressions — Conditional Filters & Filtered Measures

These are the **reusable filter conditions** and **conditional measures** that map to the Genie space's SQL Expressions feature. They cover every common condition users might ask about.

In the Genie space, add these under **Configure > Instructions > SQL Expressions** as:
- **Filters** — reusable WHERE conditions
- **Measures** — pre-filtered aggregate calculations

The cell below defines them in a format ready to add to the Genie space.

In [0]:
%sql
-- ================================================================
-- SQL EXPRESSIONS: FILTERS (Reusable WHERE conditions)
-- Add these to your Genie space under: Configure > Instructions > SQL Expressions > Filter
-- Each filter evaluates to a boolean condition.
-- ================================================================

-- ================================================================
-- FILTER: Active Rules Only
-- Synonyms: active rules, enabled rules, live rules, rules in production
-- Instructions: Use when user asks about active/enabled/live rules.
--              Excludes disabled and deactivated rules.
-- ================================================================
-- SQL Expression:
--   disabled != 'Y' AND deactivated != 'Y'

-- ================================================================
-- FILTER: Disabled Rules Only
-- Synonyms: disabled rules, turned off rules, inactive rules
-- Instructions: Use when user asks specifically about disabled rules
--              to identify missed opportunity.
-- ================================================================
-- SQL Expression:
--   disabled = 'Y'

-- ================================================================
-- FILTER: Deactivated Rules Only
-- Synonyms: deactivated rules, retired rules
-- Instructions: Use when user asks about deactivated/retired rules.
-- ================================================================
-- SQL Expression:
--   deactivated = 'Y'

-- ================================================================
-- FILTER: Medicare Only
-- Synonyms: Medicare, Medicare claims, Medicare LOB
-- Instructions: Use when user asks about Medicare specifically.
-- ================================================================
-- SQL Expression:
--   insurance_desc = 'Medicare'

-- ================================================================
-- FILTER: Medicaid Only
-- Synonyms: Medicaid, Medicaid claims, Medicaid LOB
-- Instructions: Use when user asks about Medicaid specifically.
-- ================================================================
-- SQL Expression:
--   insurance_desc = 'Medicaid'

-- ================================================================
-- FILTER: Commercial Only
-- Synonyms: Commercial, Commercial claims, Commercial LOB
-- Instructions: Use when user asks about Commercial specifically.
-- ================================================================
-- SQL Expression:
--   insurance_desc = 'Commercial'

-- ================================================================
-- FILTER: Blues Plans Only
-- Synonyms: Blues, BCBS, Blue Cross, Blue Shield plans
-- Instructions: Use when user asks about Blues/BCBS plans.
-- ================================================================
-- SQL Expression:
--   blues_indicator = 'Y'

-- ================================================================
-- FILTER: Non-Blues Plans Only
-- Synonyms: non-Blues, non-BCBS, not Blue Cross
-- Instructions: Use when user asks about non-Blues payers.
-- ================================================================
-- SQL Expression:
--   blues_indicator != 'Y'

-- ================================================================
-- FILTER: IDN Customers Only
-- Synonyms: IDN, integrated delivery network, health systems
-- Instructions: Use when user asks about IDN customers.
-- ================================================================
-- SQL Expression:
--   idn_yn = 'Y'

-- ================================================================
-- FILTER: ICM Product Only (Inpatient)
-- Synonyms: ICM, inpatient, inpatient claims
-- Instructions: Use when user asks about inpatient/ICM product.
-- ================================================================
-- SQL Expression:
--   product_type = 'ICM'

-- ================================================================
-- FILTER: ICMO Product Only (Outpatient)
-- Synonyms: ICMO, outpatient, outpatient claims
-- Instructions: Use when user asks about outpatient/ICMO product.
-- ================================================================
-- SQL Expression:
--   product_type = 'ICMO'

-- ================================================================
-- FILTER: Bundled Policies Only
-- Synonyms: bundled, policy bundles, bundle only
-- Instructions: Use when user asks about bundled policies.
-- ================================================================
-- SQL Expression:
--   policy_bundle_indicator = 'Y'

-- ================================================================
-- FILTER: Non-Bundled (Individual) Policies Only
-- Synonyms: individual policies, non-bundled, standalone policies, a la carte
-- Instructions: Use when user asks about individual/standalone policies.
-- ================================================================
-- SQL Expression:
--   policy_bundle_indicator != 'Y'

-- ================================================================
-- FILTER: Deny Edits Only
-- Synonyms: denials, deny, denied claims, deny edits
-- Instructions: Use when user asks about edits that denied claims.
-- ================================================================
-- SQL Expression:
--   final_state = 'Deny'

-- ================================================================
-- FILTER: Reduce Edits Only
-- Synonyms: reductions, reduce, reduced claims
-- Instructions: Use when user asks about edits that reduced claims.
-- ================================================================
-- SQL Expression:
--   final_state = 'Reduce'

-- ================================================================
-- FILTER: Has Unapplied Savings
-- Synonyms: has opportunity, missed savings, unapplied opportunity
-- Instructions: Use when user asks about rules/customers with missed savings.
-- ================================================================
-- SQL Expression:
--   unapplied > 0

-- ================================================================
-- FILTER: Rules with Hits
-- Synonyms: rules that fired, active edits, rules with hits
-- Instructions: Use when user wants only rules that actually triggered.
-- ================================================================
-- SQL Expression:
--   hit_flag = 1

-- ================================================================
-- FILTER: Home Blues Processing
-- Synonyms: home plan, home processing, blues home
-- Instructions: Use when user asks about Blues home plan processing.
-- ================================================================
-- SQL Expression:
--   blues_home_host = 'Home'

-- ================================================================
-- FILTER: Host Blues Processing
-- Synonyms: host plan, host processing, blues host
-- Instructions: Use when user asks about Blues host plan processing.
-- ================================================================
-- SQL Expression:
--   blues_home_host = 'Host'

-- ================================================================
-- FILTER: Cotiviti Primary Editor
-- Synonyms: primary editor, first editor, primary position
-- Instructions: Use when user asks about where Cotiviti is the primary editor.
-- ================================================================
-- SQL Expression:
--   cotiviti_edit_position = 'Primary'


SELECT 'SQL Expression filters defined above - add to Genie space' AS info

In [0]:
%sql
-- ================================================================
-- SQL EXPRESSIONS: FILTERED MEASURES
-- These are pre-computed measures with built-in conditions.
-- Add to Genie space under: Configure > Instructions > SQL Expressions > Measure
-- OR include in the metric view YAML as additional measures.
-- ================================================================

-- ================================================================
-- MEASURE: GPV for Medicare
-- Synonyms: Medicare GPV, Medicare savings, Medicare gross savings
-- Instructions: Total GPV filtered to Medicare LOB only.
-- ================================================================
-- SQL Expression:
--   SUM(CASE WHEN insurance_desc = 'Medicare' THEN COALESCE(gpv, 0) ELSE 0 END)

-- ================================================================
-- MEASURE: GPV for Commercial
-- Synonyms: Commercial GPV, Commercial savings
-- Instructions: Total GPV filtered to Commercial LOB only.
-- ================================================================
-- SQL Expression:
--   SUM(CASE WHEN insurance_desc = 'Commercial' THEN COALESCE(gpv, 0) ELSE 0 END)

-- ================================================================
-- MEASURE: GPV for Medicaid
-- Synonyms: Medicaid GPV, Medicaid savings
-- Instructions: Total GPV filtered to Medicaid LOB only.
-- ================================================================
-- SQL Expression:
--   SUM(CASE WHEN insurance_desc = 'Medicaid' THEN COALESCE(gpv, 0) ELSE 0 END)

-- ================================================================
-- MEASURE: NPV for Active Rules Only
-- Synonyms: active NPV, enabled NPV, live rule savings
-- Instructions: Net savings from active (non-disabled, non-deactivated) rules.
-- ================================================================
-- SQL Expression:
--   SUM(CASE WHEN disabled != 'Y' AND deactivated != 'Y' THEN COALESCE(npv, 0) ELSE 0 END)

-- ================================================================
-- MEASURE: GPV from Bundled Policies
-- Synonyms: bundle GPV, bundled savings, policy bundle value
-- Instructions: GPV from rules in policy bundles.
-- ================================================================
-- SQL Expression:
--   SUM(CASE WHEN policy_bundle_indicator = 'Y' THEN COALESCE(gpv, 0) ELSE 0 END)

-- ================================================================
-- MEASURE: GPV from Individual Policies
-- Synonyms: individual GPV, non-bundled GPV, a la carte savings
-- Instructions: GPV from rules NOT in bundles.
-- ================================================================
-- SQL Expression:
--   SUM(CASE WHEN policy_bundle_indicator != 'Y' THEN COALESCE(gpv, 0) ELSE 0 END)

-- ================================================================
-- MEASURE: Unapplied from Disabled Rules
-- Synonyms: disabled opportunity, disabled unapplied, missed from disabled
-- Instructions: Total unapplied value specifically from disabled rules.
--              Shows potential savings if rules were re-enabled.
-- ================================================================
-- SQL Expression:
--   SUM(CASE WHEN disabled = 'Y' THEN COALESCE(unapplied, 0) ELSE 0 END)

-- ================================================================
-- MEASURE: GPV for Deny Edits
-- Synonyms: deny GPV, denial savings, denied amount
-- Instructions: GPV from edits with Deny final state.
-- ================================================================
-- SQL Expression:
--   SUM(CASE WHEN final_state = 'Deny' THEN COALESCE(gpv, 0) ELSE 0 END)

-- ================================================================
-- MEASURE: GPV for Reduce Edits
-- Synonyms: reduce GPV, reduction savings, reduced amount
-- Instructions: GPV from edits with Reduce final state.
-- ================================================================
-- SQL Expression:
--   SUM(CASE WHEN final_state = 'Reduce' THEN COALESCE(gpv, 0) ELSE 0 END)

-- ================================================================
-- MEASURE: GPV for Blues Plans
-- Synonyms: Blues GPV, BCBS savings, Blue Cross value
-- Instructions: GPV from Blues/BCBS payers only.
-- ================================================================
-- SQL Expression:
--   SUM(CASE WHEN blues_indicator = 'Y' THEN COALESCE(gpv, 0) ELSE 0 END)

-- ================================================================
-- MEASURE: GPV for Non-Blues Plans
-- Synonyms: non-Blues GPV, non-BCBS savings
-- Instructions: GPV from non-Blues payers.
-- ================================================================
-- SQL Expression:
--   SUM(CASE WHEN blues_indicator != 'Y' THEN COALESCE(gpv, 0) ELSE 0 END)

-- ================================================================
-- MEASURE: NPV for ICM (Inpatient)
-- Synonyms: inpatient NPV, ICM savings, inpatient net savings
-- Instructions: Net savings from inpatient product only.
-- ================================================================
-- SQL Expression:
--   SUM(CASE WHEN product_type = 'ICM' THEN COALESCE(npv, 0) ELSE 0 END)

-- ================================================================
-- MEASURE: NPV for ICMO (Outpatient)
-- Synonyms: outpatient NPV, ICMO savings, outpatient net savings
-- Instructions: Net savings from outpatient product only.
-- ================================================================
-- SQL Expression:
--   SUM(CASE WHEN product_type = 'ICMO' THEN COALESCE(npv, 0) ELSE 0 END)

-- ================================================================
-- MEASURE: Edited Lines for Active Rules
-- Synonyms: active edits, live edit count, enabled edits
-- Instructions: Edit count from active rules only.
-- ================================================================
-- SQL Expression:
--   SUM(CASE WHEN disabled != 'Y' AND deactivated != 'Y' THEN COALESCE(edited_lines, 0) ELSE 0 END)

-- ================================================================
-- MEASURE: Customer Count for Blues
-- Synonyms: Blues customers, BCBS client count
-- Instructions: Number of distinct customers with Blues payers.
-- ================================================================
-- SQL Expression:
--   COUNT(DISTINCT CASE WHEN blues_indicator = 'Y' THEN customer END)

-- ================================================================
-- MEASURE: Payer Count for Medicare
-- Synonyms: Medicare payers, Medicare plans
-- Instructions: Number of distinct payers in Medicare LOB.
-- ================================================================
-- SQL Expression:
--   COUNT(DISTINCT CASE WHEN insurance_desc = 'Medicare' THEN payer_short END)

-- ================================================================
-- MEASURE: GPV% for Active Rules (Level 1)
-- Synonyms: active GPV%, enabled GPV rate, live savings rate
-- Instructions: GPV% considering only active (non-disabled) rules.
-- ================================================================
-- SQL Expression:
--   100.0 * SUM(CASE WHEN disabled != 'Y' AND deactivated != 'Y' THEN COALESCE(gpv, 0) ELSE 0 END)
--   / NULLIF(SUM(CASE WHEN disabled != 'Y' AND deactivated != 'Y' THEN COALESCE(paid, 0) ELSE 0 END), 0)

-- ================================================================
-- MEASURE: Unapplied % Opportunity
-- Synonyms: disabled opportunity rate, re-enable potential
-- Instructions: What % of total unapplied comes from disabled rules?
--              Shows how much opportunity re-enabling would capture.
-- ================================================================
-- SQL Expression:
--   100.0 * SUM(CASE WHEN disabled = 'Y' THEN COALESCE(unapplied, 0) ELSE 0 END)
--   / NULLIF(SUM(COALESCE(unapplied, 0)), 0)


SELECT 'SQL Expression measures defined above - add to Genie space' AS info

In [0]:
%sql
-- ================================================================
-- SQL EXPRESSIONS: DIMENSIONS (Derived/Computed)
-- Add to Genie space under: Configure > Instructions > SQL Expressions > Dimension
-- Each expression alters the value of each row for cleaner display.
-- ================================================================

-- ================================================================
-- DIMENSION: LOB Category
-- Synonyms: LOB group, insurance category, coverage group
-- Instructions: Simplified LOB grouping into main categories.
-- ================================================================
-- SQL Expression:
--   CASE
--     WHEN insurance_desc IN ('Commercial', 'Self-Funded') THEN 'Commercial'
--     WHEN insurance_desc = 'Medicare' THEN 'Medicare'
--     WHEN insurance_desc = 'Medicaid' THEN 'Medicaid'
--     WHEN insurance_desc = 'FEP' THEN 'FEP'
--     ELSE 'Other'
--   END

-- ================================================================
-- DIMENSION: Rule Status Category
-- Synonyms: rule state, active/inactive, rule lifecycle
-- Instructions: Combines disabled and deactivated into a single status.
-- ================================================================
-- SQL Expression:
--   CASE
--     WHEN disabled = 'Y' THEN 'Disabled'
--     WHEN deactivated = 'Y' THEN 'Deactivated'
--     ELSE 'Active'
--   END

-- ================================================================
-- DIMENSION: Payer Size Tier
-- Synonyms: payer tier, plan size, volume tier
-- Instructions: Categorizes payers by their paid volume.
--              Requires context-specific thresholds.
-- ================================================================
-- SQL Expression (example thresholds):
--   CASE
--     WHEN paid > 1000000000 THEN 'Large (>$1B)'
--     WHEN paid > 100000000 THEN 'Medium ($100M-$1B)'
--     ELSE 'Small (<$100M)'
--   END

-- ================================================================
-- DIMENSION: Edit Position Category
-- Synonyms: processing order, editor ranking
-- Instructions: Simplified edit position grouping.
-- ================================================================
-- SQL Expression:
--   CASE
--     WHEN cotiviti_edit_position = 'Primary' THEN 'Cotiviti Primary'
--     WHEN cotiviti_edit_position = 'Secondary' THEN 'Cotiviti Secondary'
--     ELSE cotiviti_edit_position
--   END

-- ================================================================
-- DIMENSION: Has Competitor
-- Synonyms: competitive account, third party present
-- Instructions: Whether a third-party editor is present.
-- ================================================================
-- SQL Expression:
--   CASE
--     WHEN primary_3rd_party_editor IS NOT NULL
--          AND primary_3rd_party_editor != ''
--     THEN 'Has Competitor'
--     ELSE 'No Competitor'
--   END

-- ================================================================
-- DIMENSION: Bundle Status
-- Synonyms: packaging status, bundle/individual
-- Instructions: Whether the policy is bundled, packaged, or individual.
-- ================================================================
-- SQL Expression:
--   CASE
--     WHEN policy_bundle_indicator = 'Y' THEN 'Bundled'
--     WHEN policy_package_indicator = 'Y' THEN 'Packaged'
--     WHEN policy_package_individual_indicator = 'Y' THEN 'Individual'
--     ELSE 'Unclassified'
--   END

-- ================================================================
-- DIMENSION: Invoice Recency
-- Synonyms: data freshness, how recent, invoice age
-- Instructions: Categorizes records by how recent the invoice is.
-- ================================================================
-- SQL Expression:
--   CASE
--     WHEN max_invoice_year = CAST(YEAR(CURRENT_DATE()) AS STRING) THEN 'Current Year'
--     WHEN max_invoice_year = CAST(YEAR(CURRENT_DATE()) - 1 AS STRING) THEN 'Prior Year'
--     ELSE 'Older'
--   END


SELECT 'SQL Expression dimensions defined above - add to Genie space' AS info

## ⚠️ Critical: LOD-Safe Denominator Architecture

### Why This Is Hard

A metric view has **ONE fixed expression per measure**. But the LOD-safe pattern in your Genie space **adapts its denominator based on the query context** (CTE + window function approach). These are fundamentally different:

| Approach | How Denominator Works |
| --- | --- |
| **Genie Example SQL** | CTE groups by (target_dim, payer, state), then WINDOW computes paid at (payer, state) level. Adapts per query. |
| **Metric View** | Single `expr` evaluated after GROUP BY. Same formula regardless of which dimension you slice by. |

### The Problem with the Source-Level Window

The current source uses:
```
SUM(paid) OVER (PARTITION BY payer_short, final_state) AS paid_lod_payer_state
```

This is **incorrect** because it sums `paid` across ALL rows in that (payer, state) partition — including duplicates from multiple DPs. If `paid` is repeated per DP row, this amplifies the overcounting rather than fixing it.

### What We Need to Validate

Before finalizing the LOD-safe source, we need to understand the **exact grain of `paid`**:

| Scenario | Implication |
| --- | --- |
| A) `paid` is the SAME value repeated on every DP row for a (customer, payer, state) | Need to de-duplicate before windowing |
| B) `paid` is unique per (customer, payer, state, dp_key) — each DP gets its own portion | Standard SUM works, but LOD means using total payer paid as denominator |

### Correct Architecture (Once Grain Is Confirmed)

**Step 1:** Create an intermediate view that de-duplicates `paid` at its true grain:
```sql
CREATE OR REPLACE VIEW ...ppm_benchmark_lod_source AS
WITH paid_unique AS (
  SELECT customer, payer_short, final_state,
         MAX(paid) AS paid_at_grain  -- or SUM if paid is portioned
  FROM ppm_benchmark
  WHERE customer IS NOT NULL AND customer != 'Lib'
  GROUP BY customer, payer_short, final_state
)
SELECT t.*, 
       pu.paid_at_grain AS paid_lod_safe
FROM ppm_benchmark t
JOIN paid_unique pu
  ON t.customer = pu.customer
  AND t.payer_short = pu.payer_short
  AND t.final_state = pu.final_state
WHERE t.customer IS NOT NULL AND t.customer != 'Lib'
```

**Step 2:** Use that view as the metric view source

**Step 3:** LOD-safe measures use `paid_lod_safe` as denominator:
```yaml
- name: GPV Percent LOD Safe
  expr: 100.0 * SUM(COALESCE(gpv, 0)) / NULLIF(SUM(paid_lod_safe), 0)
```

### Validation Queries (Run These First!)
```sql
-- Check: Is paid the same for all DPs within (customer, payer, state)?
SELECT customer, payer_short, final_state, 
       COUNT(DISTINCT paid) AS distinct_paid_values,
       COUNT(DISTINCT dp_key) AS distinct_dps
FROM ppm_benchmark
WHERE customer IS NOT NULL AND customer != 'Lib'
GROUP BY customer, payer_short, final_state
HAVING COUNT(DISTINCT paid) > 1
LIMIT 10;

-- Check: How many rows per (customer, payer, state, dp_key)?
SELECT customer, payer_short, final_state, dp_key, COUNT(*) as row_count
FROM ppm_benchmark
WHERE customer IS NOT NULL AND customer != 'Lib'
GROUP BY customer, payer_short, final_state, dp_key
HAVING COUNT(*) > 1
LIMIT 10;
```

Once we know the grain, we can finalize the correct LOD-safe source.

In [0]:
%sql
-- VALIDATION QUERY 1: Is paid the same for all DPs within (customer, payer, state)?
-- If this returns rows, then paid varies by DP (Scenario B: portioned)
-- If this returns ZERO rows, then paid is duplicated identically (Scenario A: repeated)
SELECT customer, payer_short, final_state, 
       COUNT(DISTINCT paid) AS distinct_paid_values,
       COUNT(DISTINCT dp_key) AS distinct_dps,
       MIN(paid) AS min_paid,
       MAX(paid) AS max_paid
FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark
WHERE customer IS NOT NULL AND customer != 'Lib'
GROUP BY customer, payer_short, final_state
HAVING COUNT(DISTINCT paid) > 1
LIMIT 20

In [0]:
%sql
-- VALIDATION QUERY 2: What is the actual grain of the table?
-- How many rows exist per (customer, payer, state, dp_key)?
SELECT 
  CASE 
    WHEN row_count = 1 THEN '1 row (dp_key is lowest grain)'
    ELSE CONCAT(CAST(row_count AS STRING), ' rows (sub-rule grain below dp_key)')
  END AS grain_level,
  COUNT(*) AS num_combinations
FROM (
  SELECT customer, payer_short, final_state, dp_key, COUNT(*) as row_count
  FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark
  WHERE customer IS NOT NULL AND customer != 'Lib'
  GROUP BY customer, payer_short, final_state, dp_key
)
GROUP BY 1
ORDER BY 1

In [0]:
%sql
-- VALIDATION QUERY 3: Compare simple vs LOD-safe GPV% for a known DP
-- This compares against the Genie's known-correct LOD-safe answer
WITH agg AS (
  SELECT
    payer_short AS ps,
    final_state AS fs,
    SUM(COALESCE(gpv, 0)) AS gpv_val,
    SUM(COALESCE(paid, 0)) AS paid_val
  FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark
  WHERE dp_key = '8290'
    AND customer IS NOT NULL AND customer != 'Lib'
  GROUP BY payer_short, final_state
),
with_paid_lod AS (
  SELECT
    gpv_val,
    paid_val,
    SUM(paid_val) OVER (PARTITION BY ps, fs) AS paid_lod
  FROM agg
)
SELECT 
  'DP 8290' AS dp,
  ROUND(100.0 * SUM(gpv_val) / NULLIF(SUM(paid_val), 0), 4) AS gpv_pct_simple,
  ROUND(100.0 * SUM(gpv_val) / NULLIF(SUM(paid_lod), 0), 4) AS gpv_pct_lod_safe
FROM with_paid_lod

## Metric Definitions — Layman's Explanation

### Dollar Metrics ("How much money?")
| Metric | Plain English | Formula |
| --- | --- | --- |
| **Total Paid** | All money that flowed through the claims system | SUM of paid amounts |
| **Total GPV** | Raw savings flagged by Cotiviti rules (before disputes) | SUM of gross payment value |
| **Total Adjustments** | Money that was clawed back (disputes, overrides) — usually negative | SUM of adjustments |
| **Total NPV** | Final real savings the customer keeps (GPV + Adjustments) | SUM of net payment value |
| **Total Unapplied** | Savings the customer *could* have gotten but didn't configure | SUM of unapplied value |

### Percentage Metrics ("What's the rate?")
| Metric | Plain English | Formula |
| --- | --- | --- |
| **GPV%** | Cents saved per dollar processed (gross) | GPV / Paid × 100 |
| **NPV%** | Cents saved per dollar processed (net, after disputes) | NPV / Paid × 100 |
| **APV%** | How much of GPV got reversed/disputed | Adjustments / GPV × 100 |
| **Unapplied%** | How much potential savings is being left on the table | Unapplied / (Unapplied+GPV) × 100 |
| **Adjusted Lines %** | Of flagged lines, what % actually resulted in a payment change | Adjusted Lines / Edited Lines × 100 |

### Volume Metrics ("How many lines?")
| Metric | Plain English | Formula |
| --- | --- | --- |
| **Total Lines** | All claim lines that went through the system | SUM of total_lines |
| **Edited Lines** | Lines where at least one rule fired | SUM of edited_lines |
| **Adjusted Lines** | Lines where the edit actually changed the payment | SUM of adjusted_lines |
| **Edits per 1,000** | How many lines get flagged per 1,000 processed (edit frequency) | Edited / Total × 1000 |

### Value Metrics ("What's each edit worth?")
| Metric | Plain English | Formula |
| --- | --- | --- |
| **Original Paid per Edit** | Average dollar value of lines that got edited | Paid (hit lines) / Edited Lines (hit lines) |
| **Original Paid per TBA Line** | Average dollar value of any line flowing through | Total Paid / Total Lines |

### Count Metrics ("How many entities?")
| Metric | Plain English | Formula |
| --- | --- | --- |
| **Customer Count** | Number of distinct real customers | COUNT DISTINCT customer |
| **Payer Count** | Number of distinct real payers/plans | COUNT DISTINCT payer_short |

---

### Dimension Groups ("How can I slice the data?")
| Group | Dimensions | Use Case |
| --- | --- | --- |
| **Business** | Customer, Customer Segment, LOB, Product Type, Claim Type | "Show me GPV% for Medicare" |
| **Payer** | Payer, Super Payer, Payer State, Blues Indicator | "Which payers have highest NPV%?" |
| **Rule/Policy** | Decision Point, Medical Policy, Topic, Sub Rule, Mid Rule | "GPV% for DP 8290" |
| **Status** | Final State, Status, Flag, Solution, Bundle/Package indicators | "Breakdown by edit outcome" |
| **Team** | SVP, CEL, CSD, CSM, BA, Claims Platform | "NPV% by client engagement lead" |

# Example Queries — Complete Coverage

Below are ALL query patterns across every level and combination users could ask.
These serve as:
1. **Genie Example SQL Queries** — copy into your Genie space under Configure > Instructions > Example SQL
2. **Validation reference** — verify metric view produces correct results
3. **Documentation** — shows which measure to use at which level

## Query Pattern Index

| # | Pattern | Level | Measures Used |
| --- | --- | --- | --- |
| 1 | Overall Benchmarks (all 16 metrics) | Overall | Standard |
| 2 | By Customer | Level 1 | Standard |
| 3 | By Line of Business (LOB) | Level 1 | Standard |
| 4 | By Payer | Level 1 | Standard |
| 5 | By Final State | Level 1 | Standard |
| 6 | By Product Type | Level 1 | Standard |
| 7 | By Super Payer | Level 1 | Standard |
| 8 | By Payer State | Level 1 | Standard |
| 9 | By Decision Point (DP) | Level 2 | LOD-Safe |
| 10 | By Medical Policy | Level 2 | LOD-Safe |
| 11 | By Topic | Level 2 | LOD-Safe |
| 12 | By Sub Rule | Level 2 | LOD-Safe |
| 13 | DP × LOB (cross-dimensional) | Level 2 × 1 | LOD-Safe |
| 14 | Medical Policy × Customer | Level 2 × 1 | LOD-Safe |
| 15 | Customer × LOB (multi Level 1) | Level 1 × 1 | Standard |
| 16 | Customer × Payer (drilldown) | Level 1 × 1 | Standard |
| 17 | Filtered: Specific Customer | Level 1 filtered | Standard |
| 18 | Filtered: Specific DP | Level 2 filtered | LOD-Safe |
| 19 | Filtered: Specific Medical Policy | Level 2 filtered | LOD-Safe |
| 20 | Adoption by DP | Level 2 | Adoption |
| 21 | Adoption by Medical Policy × LOB | Level 2 × 1 | Adoption |
| 22 | Top-N DPs by NPV | Level 2 ranked | LOD-Safe |
| 23 | Top-N Customers by GPV | Level 1 ranked | Standard |
| 24 | Concentration: DP share of total | Level 2 | Concentration |
| 25 | Team: Metrics by SVP | Level 1 | Standard |
| 26 | Team: Metrics by CEL | Level 1 | Standard |
| 27 | Platform/Competitor analysis | Level 1 | Standard |
| 28 | Blues vs Non-Blues | Level 1 | Standard |
| 29 | Disabled/Deactivated rules | Level 1 filtered | Standard |
| 30 | Unapplied opportunity by Customer | Level 1 | Derived |

In [0]:
%sql
-- ================================================================
-- QUERY 1: Overall Benchmarks — All 16 standard metrics at once
-- Question: "Show me benchmark values" / "What are the overall metrics?"
-- Level: Overall (no GROUP BY)
-- ================================================================
SELECT
  MEASURE(`Customer Count`) AS customer_count,
  MEASURE(`Payer Count`) AS payer_count,
  MEASURE(`GPV Percent`) AS gpv_pct,
  MEASURE(`APV Percent`) AS apv_pct,
  MEASURE(`NPV Percent`) AS npv_pct,
  MEASURE(`Unapplied Percent`) AS unapplied_pct,
  MEASURE(`Total Paid`) AS total_paid,
  MEASURE(`Total GPV`) AS total_gpv,
  MEASURE(`Total Adjustments`) AS total_adjustments,
  MEASURE(`Total NPV`) AS total_npv,
  MEASURE(`Total Unapplied`) AS total_unapplied,
  MEASURE(`Adjusted Lines`) AS adjusted_lines,
  MEASURE(`Adjusted Lines Percent`) AS adjusted_lines_pct,
  MEASURE(`Edits per 1000`) AS edits_per_1000,
  MEASURE(`Original Paid per Edit`) AS paid_per_edit,
  MEASURE(`Original Paid per TBA Line`) AS paid_per_tba_line
FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark_metrics

In [0]:
%sql
-- ================================================================
-- QUERY 2: All metrics by Customer
-- Question: "Show me metrics by customer" / "GPV% by client"
-- Level: Level 1 — Standard measures correct
-- ================================================================
SELECT
  `Customer`,
  MEASURE(`GPV Percent`) AS gpv_pct,
  MEASURE(`NPV Percent`) AS npv_pct,
  MEASURE(`APV Percent`) AS apv_pct,
  MEASURE(`Unapplied Percent`) AS unapplied_pct,
  MEASURE(`Total Paid`) AS total_paid,
  MEASURE(`Total GPV`) AS total_gpv,
  MEASURE(`Total NPV`) AS total_npv,
  MEASURE(`Total Unapplied`) AS total_unapplied,
  MEASURE(`Edits per 1000`) AS edits_per_1000,
  MEASURE(`Adjusted Lines Percent`) AS adj_lines_pct,
  MEASURE(`Original Paid per Edit`) AS paid_per_edit,
  MEASURE(`Payer Count`) AS payer_count,
  MEASURE(`DP Count`) AS dp_count
FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark_metrics
GROUP BY `Customer`
ORDER BY MEASURE(`Total Paid`) DESC

In [0]:
%sql
-- ================================================================
-- QUERY 3: All metrics by Line of Business (LOB)
-- Question: "GPV% by LOB" / "Breakdown by insurance type"
-- Level: Level 1 — Standard measures correct
-- ================================================================
SELECT
  `Line of Business`,
  MEASURE(`GPV Percent`) AS gpv_pct,
  MEASURE(`NPV Percent`) AS npv_pct,
  MEASURE(`APV Percent`) AS apv_pct,
  MEASURE(`Unapplied Percent`) AS unapplied_pct,
  MEASURE(`Total Paid`) AS total_paid,
  MEASURE(`Total GPV`) AS total_gpv,
  MEASURE(`Total NPV`) AS total_npv,
  MEASURE(`Edits per 1000`) AS edits_per_1000,
  MEASURE(`Adjusted Lines Percent`) AS adj_lines_pct,
  MEASURE(`Customer Count`) AS customer_count,
  MEASURE(`Payer Count`) AS payer_count
FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark_metrics
GROUP BY `Line of Business`
ORDER BY MEASURE(`Total Paid`) DESC

In [0]:
%sql
-- ================================================================
-- QUERY 4: All metrics by Payer
-- Question: "Show me metrics by payer" / "Which payers have highest NPV%?"
-- Level: Level 1 — Standard measures correct
-- ================================================================
SELECT
  `Payer`,
  `Payer Full Name`,
  `Super Payer`,
  MEASURE(`GPV Percent`) AS gpv_pct,
  MEASURE(`NPV Percent`) AS npv_pct,
  MEASURE(`APV Percent`) AS apv_pct,
  MEASURE(`Total Paid`) AS total_paid,
  MEASURE(`Total GPV`) AS total_gpv,
  MEASURE(`Total NPV`) AS total_npv,
  MEASURE(`Edits per 1000`) AS edits_per_1000,
  MEASURE(`DP Count`) AS dp_count
FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark_metrics
GROUP BY `Payer`, `Payer Full Name`, `Super Payer`
ORDER BY MEASURE(`Total Paid`) DESC

In [0]:
%sql
-- ================================================================
-- QUERY 5: All metrics by Final State
-- Question: "Breakdown by edit outcome" / "Metrics by disposition"
-- Level: Level 1 — Standard measures correct
-- ================================================================
SELECT
  `Final State`,
  MEASURE(`GPV Percent`) AS gpv_pct,
  MEASURE(`NPV Percent`) AS npv_pct,
  MEASURE(`Total Paid`) AS total_paid,
  MEASURE(`Total GPV`) AS total_gpv,
  MEASURE(`Total NPV`) AS total_npv,
  MEASURE(`Edited Lines`) AS edited_lines,
  MEASURE(`Adjusted Lines`) AS adjusted_lines,
  MEASURE(`Adjusted Lines Percent`) AS adj_lines_pct
FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark_metrics
GROUP BY `Final State`
ORDER BY MEASURE(`Total GPV`) DESC

In [0]:
%sql
-- ================================================================
-- QUERY 6: By Product Type
-- Question: "GPV% by product type" / "ICM vs ICMO comparison"
-- Level: Level 1 — Standard measures correct
-- ================================================================
SELECT
  `Product Type`,
  MEASURE(`GPV Percent`) AS gpv_pct,
  MEASURE(`NPV Percent`) AS npv_pct,
  MEASURE(`Total Paid`) AS total_paid,
  MEASURE(`Total GPV`) AS total_gpv,
  MEASURE(`Total NPV`) AS total_npv,
  MEASURE(`Edits per 1000`) AS edits_per_1000,
  MEASURE(`Customer Count`) AS customer_count
FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark_metrics
GROUP BY `Product Type`
ORDER BY MEASURE(`Total Paid`) DESC

In [0]:
%sql
-- ================================================================
-- QUERY 7: By Super Payer
-- Question: "Metrics by parent payer" / "Super payer performance"
-- Level: Level 1 — Standard measures correct
-- ================================================================
SELECT
  `Super Payer`,
  `Super Payer Full Name`,
  MEASURE(`GPV Percent`) AS gpv_pct,
  MEASURE(`NPV Percent`) AS npv_pct,
  MEASURE(`Total Paid`) AS total_paid,
  MEASURE(`Total NPV`) AS total_npv,
  MEASURE(`Payer Count`) AS payer_count,
  MEASURE(`Customer Count`) AS customer_count
FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark_metrics
GROUP BY `Super Payer`, `Super Payer Full Name`
ORDER BY MEASURE(`Total Paid`) DESC

In [0]:
%sql
-- ================================================================
-- QUERY 8: By Payer State
-- Question: "GPV% by state" / "Geographic breakdown"
-- Level: Level 1 — Standard measures correct
-- ================================================================
SELECT
  `Payer State`,
  MEASURE(`GPV Percent`) AS gpv_pct,
  MEASURE(`NPV Percent`) AS npv_pct,
  MEASURE(`Total Paid`) AS total_paid,
  MEASURE(`Total NPV`) AS total_npv,
  MEASURE(`Customer Count`) AS customer_count,
  MEASURE(`Payer Count`) AS payer_count
FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark_metrics
GROUP BY `Payer State`
ORDER BY MEASURE(`Total Paid`) DESC

In [0]:
%sql
-- ================================================================
-- QUERY 9: By Decision Point (DP) — LOD-SAFE REQUIRED
-- Question: "GPV% by DP" / "Show me metrics for each rule"
-- Level: Level 2 — MUST use LOD-safe measures
-- ================================================================
SELECT
  `Decision Point`,
  `Decision Point Description`,
  MEASURE(`GPV Percent LOD Safe`) AS gpv_pct,
  MEASURE(`NPV Percent LOD Safe`) AS npv_pct,
  MEASURE(`APV Percent`) AS apv_pct,
  MEASURE(`Total GPV`) AS total_gpv,
  MEASURE(`Total NPV`) AS total_npv,
  MEASURE(`Total Unapplied`) AS total_unapplied,
  MEASURE(`Edits per 1000`) AS edits_per_1000,
  MEASURE(`Adjusted Lines Percent`) AS adj_lines_pct,
  MEASURE(`Customer Count`) AS customer_count,
  MEASURE(`Customer Adoption Rate`) AS adoption_rate,
  MEASURE(`GPV Concentration`) AS gpv_share
FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark_metrics
GROUP BY `Decision Point`, `Decision Point Description`
ORDER BY MEASURE(`Total GPV`) DESC

In [0]:
%sql
-- ================================================================
-- QUERY 10: By Medical Policy — LOD-SAFE REQUIRED
-- Question: "GPV% by medical policy" / "Which policies save the most?"
-- Level: Level 2 — MUST use LOD-safe measures
-- ================================================================
SELECT
  `Medical Policy`,
  MEASURE(`GPV Percent LOD Safe`) AS gpv_pct,
  MEASURE(`NPV Percent LOD Safe`) AS npv_pct,
  MEASURE(`APV Percent`) AS apv_pct,
  MEASURE(`Total GPV`) AS total_gpv,
  MEASURE(`Total NPV`) AS total_npv,
  MEASURE(`DP Count`) AS dp_count,
  MEASURE(`Customer Count`) AS customer_count,
  MEASURE(`Customer Adoption Rate`) AS adoption_rate,
  MEASURE(`GPV Concentration`) AS gpv_share
FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark_metrics
GROUP BY `Medical Policy`
ORDER BY MEASURE(`Total GPV`) DESC

In [0]:
%sql
-- ================================================================
-- QUERY 11: By Topic — LOD-SAFE REQUIRED
-- Question: "GPV% by topic" / "Which clinical categories save the most?"
-- Level: Level 2 — MUST use LOD-safe measures
-- ================================================================
SELECT
  `Topic`,
  MEASURE(`GPV Percent LOD Safe`) AS gpv_pct,
  MEASURE(`NPV Percent LOD Safe`) AS npv_pct,
  MEASURE(`APV Percent`) AS apv_pct,
  MEASURE(`Total GPV`) AS total_gpv,
  MEASURE(`Total NPV`) AS total_npv,
  MEASURE(`DP Count`) AS dp_count,
  MEASURE(`Medical Policy Count`) AS policy_count,
  MEASURE(`Customer Adoption Rate`) AS adoption_rate
FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark_metrics
GROUP BY `Topic`
ORDER BY MEASURE(`Total GPV`) DESC

In [0]:
%sql
-- ================================================================
-- QUERY 12: By Sub Rule — LOD-SAFE REQUIRED
-- Question: "Metrics by sub-rule" / "Which sub-rules drive most savings?"
-- Level: Level 2 — MUST use LOD-safe measures
-- ================================================================
SELECT
  `Sub Rule Key`,
  `Sub Rule Description`,
  `DP for Sub Rule`,
  `Topic for Sub Rule`,
  `Medical Policy for Sub Rule`,
  MEASURE(`GPV Percent LOD Safe`) AS gpv_pct,
  MEASURE(`NPV Percent LOD Safe`) AS npv_pct,
  MEASURE(`Total GPV`) AS total_gpv,
  MEASURE(`Total NPV`) AS total_npv,
  MEASURE(`Customer Count`) AS customer_count
FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark_metrics
GROUP BY `Sub Rule Key`, `Sub Rule Description`, `DP for Sub Rule`,
         `Topic for Sub Rule`, `Medical Policy for Sub Rule`
ORDER BY MEASURE(`Total GPV`) DESC

In [0]:
%sql
-- ================================================================
-- QUERY 13: DP × LOB (cross-dimensional)
-- Question: "GPV% for DP 8290 by LOB" / "How does this rule perform by LOB?"
-- Level: Level 2 × Level 1 — LOD-safe required (DP is Level 2)
-- ================================================================
SELECT
  `Decision Point`,
  `Decision Point Description`,
  `Line of Business`,
  MEASURE(`GPV Percent LOD Safe`) AS gpv_pct,
  MEASURE(`NPV Percent LOD Safe`) AS npv_pct,
  MEASURE(`Total GPV`) AS total_gpv,
  MEASURE(`Total NPV`) AS total_npv,
  MEASURE(`Customer Count`) AS customer_count
FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark_metrics
GROUP BY `Decision Point`, `Decision Point Description`, `Line of Business`
ORDER BY `Decision Point`, MEASURE(`Total GPV`) DESC

In [0]:
%sql
-- ================================================================
-- QUERY 14: Medical Policy × Customer (cross-dimensional)
-- Question: "Which customers use cardiology policy?" / "Policy by customer"
-- Level: Level 2 × Level 1 — LOD-safe required
-- ================================================================
SELECT
  `Medical Policy`,
  `Customer`,
  MEASURE(`GPV Percent LOD Safe`) AS gpv_pct,
  MEASURE(`NPV Percent LOD Safe`) AS npv_pct,
  MEASURE(`Total GPV`) AS total_gpv,
  MEASURE(`Total NPV`) AS total_npv,
  MEASURE(`DP Count`) AS dp_count
FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark_metrics
GROUP BY `Medical Policy`, `Customer`
ORDER BY `Medical Policy`, MEASURE(`Total GPV`) DESC

In [0]:
%sql
-- ================================================================
-- QUERY 15: Customer × LOB (multi Level 1)
-- Question: "GPV% by customer and LOB" / "Customer performance by LOB"
-- Level: Level 1 × Level 1 — Standard measures correct
-- ================================================================
SELECT
  `Customer`,
  `Line of Business`,
  MEASURE(`GPV Percent`) AS gpv_pct,
  MEASURE(`NPV Percent`) AS npv_pct,
  MEASURE(`Total Paid`) AS total_paid,
  MEASURE(`Total GPV`) AS total_gpv,
  MEASURE(`Total NPV`) AS total_npv,
  MEASURE(`Edits per 1000`) AS edits_per_1000
FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark_metrics
GROUP BY `Customer`, `Line of Business`
ORDER BY `Customer`, MEASURE(`Total Paid`) DESC

In [0]:
%sql
-- ================================================================
-- QUERY 16: Customer × Payer (drilldown)
-- Question: "Show me payers for customer X" / "Payer drilldown"
-- Level: Level 1 × Level 1 — Standard measures correct
-- ================================================================
SELECT
  `Customer`,
  `Payer`,
  `Payer Full Name`,
  `Payer State`,
  `Blues Indicator`,
  MEASURE(`GPV Percent`) AS gpv_pct,
  MEASURE(`NPV Percent`) AS npv_pct,
  MEASURE(`Total Paid`) AS total_paid,
  MEASURE(`Total GPV`) AS total_gpv,
  MEASURE(`Total NPV`) AS total_npv
FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark_metrics
GROUP BY `Customer`, `Payer`, `Payer Full Name`, `Payer State`, `Blues Indicator`
ORDER BY `Customer`, MEASURE(`Total Paid`) DESC

In [0]:
%sql
-- ================================================================
-- QUERY 17: Filtered — Specific Customer
-- Question: "Show me metrics for [Customer Name]" / "How is Aetna performing?"
-- Level: Level 1 filtered — Standard measures correct
-- ================================================================
SELECT
  `Customer`,
  `Line of Business`,
  MEASURE(`GPV Percent`) AS gpv_pct,
  MEASURE(`NPV Percent`) AS npv_pct,
  MEASURE(`APV Percent`) AS apv_pct,
  MEASURE(`Total Paid`) AS total_paid,
  MEASURE(`Total GPV`) AS total_gpv,
  MEASURE(`Total NPV`) AS total_npv,
  MEASURE(`Unapplied Percent`) AS unapplied_pct,
  MEASURE(`Edits per 1000`) AS edits_per_1000,
  MEASURE(`Adjusted Lines Percent`) AS adj_lines_pct,
  MEASURE(`Original Paid per Edit`) AS paid_per_edit,
  MEASURE(`Payer Count`) AS payer_count,
  MEASURE(`DP Count`) AS dp_count
FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark_metrics
WHERE `Customer` = '<customer_name>'
GROUP BY `Customer`, `Line of Business`
ORDER BY MEASURE(`Total Paid`) DESC

In [0]:
%sql
-- ================================================================
-- QUERY 18: Filtered — Specific DP (LOD-SAFE)
-- Question: "Show me GPV% for DP 8290" / "How does rule 8290 perform?"
-- Level: Level 2 filtered — LOD-safe required
-- ================================================================
SELECT
  `Decision Point`,
  `Decision Point Description`,
  `Line of Business`,
  MEASURE(`GPV Percent LOD Safe`) AS gpv_pct,
  MEASURE(`NPV Percent LOD Safe`) AS npv_pct,
  MEASURE(`APV Percent`) AS apv_pct,
  MEASURE(`Total GPV`) AS total_gpv,
  MEASURE(`Total NPV`) AS total_npv,
  MEASURE(`Edits per 1000`) AS edits_per_1000,
  MEASURE(`Customer Count`) AS customer_count,
  MEASURE(`Customer Adoption Rate`) AS adoption_rate
FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark_metrics
WHERE `Decision Point` = '8290'
GROUP BY `Decision Point`, `Decision Point Description`, `Line of Business`
ORDER BY MEASURE(`Total GPV`) DESC

In [0]:
%sql
-- ================================================================
-- QUERY 19: Filtered — Specific Medical Policy (LOD-SAFE)
-- Question: "Show me metrics for cardiology policy" / "Policy performance"
-- Level: Level 2 filtered — LOD-safe required
-- NOTE: Use ILIKE for fuzzy matching on policy names
-- ================================================================
SELECT
  `Medical Policy`,
  `Customer`,
  MEASURE(`GPV Percent LOD Safe`) AS gpv_pct,
  MEASURE(`NPV Percent LOD Safe`) AS npv_pct,
  MEASURE(`Total GPV`) AS total_gpv,
  MEASURE(`Total NPV`) AS total_npv,
  MEASURE(`DP Count`) AS dp_count,
  MEASURE(`Customer Adoption Rate`) AS adoption_rate
FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark_metrics
WHERE `Medical Policy` ILIKE '%cardiology%'
GROUP BY `Medical Policy`, `Customer`
ORDER BY MEASURE(`Total GPV`) DESC

In [0]:
%sql
-- ================================================================
-- QUERY 20: Adoption by DP
-- Question: "Customer adoption rate for each DP" / "Which rules are most adopted?"
-- Level: Level 2 — Adoption metrics
-- ================================================================
SELECT
  `Decision Point`,
  `Decision Point Description`,
  `Topic`,
  `Medical Policy`,
  MEASURE(`Customer Adoption Rate`) AS customer_adoption_pct,
  MEASURE(`Payer Adoption Rate`) AS payer_adoption_pct,
  MEASURE(`Customer Count`) AS customers_using,
  MEASURE(`Payer Count`) AS payers_using,
  MEASURE(`GPV Percent LOD Safe`) AS gpv_pct,
  MEASURE(`Total GPV`) AS total_gpv
FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark_metrics
GROUP BY `Decision Point`, `Decision Point Description`, `Topic`, `Medical Policy`
ORDER BY MEASURE(`Customer Adoption Rate`) DESC

In [0]:
%sql
-- ================================================================
-- QUERY 21: Adoption by Medical Policy × LOB
-- Question: "Adoption rate for cardiology by LOB" / "Which LOBs use this policy?"
-- Level: Level 2 × Level 1 — Adoption metrics
-- ================================================================
SELECT
  `Medical Policy`,
  `Line of Business`,
  MEASURE(`Customer Adoption Rate`) AS customer_adoption_pct,
  MEASURE(`Customer Count`) AS customers_using,
  MEASURE(`GPV Percent LOD Safe`) AS gpv_pct,
  MEASURE(`NPV Percent LOD Safe`) AS npv_pct,
  MEASURE(`Total GPV`) AS total_gpv
FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark_metrics
GROUP BY `Medical Policy`, `Line of Business`
ORDER BY `Medical Policy`, MEASURE(`Customer Count`) DESC

In [0]:
%sql
-- ================================================================
-- QUERY 22: Top-N DPs by NPV (ranked)
-- Question: "Top 10 DPs by net savings" / "Highest impact rules"
-- Level: Level 2 ranked — LOD-safe required
-- ================================================================
SELECT
  `Decision Point`,
  `Decision Point Description`,
  `Topic`,
  MEASURE(`NPV Percent LOD Safe`) AS npv_pct,
  MEASURE(`GPV Percent LOD Safe`) AS gpv_pct,
  MEASURE(`Total NPV`) AS total_npv,
  MEASURE(`Total GPV`) AS total_gpv,
  MEASURE(`Customer Adoption Rate`) AS adoption_rate,
  MEASURE(`NPV Concentration`) AS npv_share_of_total
FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark_metrics
GROUP BY `Decision Point`, `Decision Point Description`, `Topic`
ORDER BY MEASURE(`Total NPV`) DESC
LIMIT 10

In [0]:
%sql
-- ================================================================
-- QUERY 23: Top-N Customers by GPV (ranked)
-- Question: "Top customers by savings" / "Largest accounts by GPV"
-- Level: Level 1 ranked — Standard measures correct
-- ================================================================
SELECT
  `Customer`,
  `Customer Segment`,
  MEASURE(`GPV Percent`) AS gpv_pct,
  MEASURE(`NPV Percent`) AS npv_pct,
  MEASURE(`Total Paid`) AS total_paid,
  MEASURE(`Total GPV`) AS total_gpv,
  MEASURE(`Total NPV`) AS total_npv,
  MEASURE(`GPV Concentration`) AS gpv_share_of_total,
  MEASURE(`Payer Count`) AS payer_count
FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark_metrics
GROUP BY `Customer`, `Customer Segment`
ORDER BY MEASURE(`Total GPV`) DESC
LIMIT 10

In [0]:
%sql
-- ================================================================
-- QUERY 24: Concentration — DP share of total
-- Question: "Which DPs account for most of total GPV?"
-- Level: Level 2 — Concentration measures
-- ================================================================
SELECT
  `Decision Point`,
  `Decision Point Description`,
  MEASURE(`GPV Concentration`) AS gpv_share_pct,
  MEASURE(`NPV Concentration`) AS npv_share_pct,
  MEASURE(`Total GPV`) AS total_gpv,
  MEASURE(`Total NPV`) AS total_npv,
  MEASURE(`Customer Count`) AS customer_count
FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark_metrics
GROUP BY `Decision Point`, `Decision Point Description`
ORDER BY MEASURE(`GPV Concentration`) DESC
LIMIT 20

In [0]:
%sql
-- ================================================================
-- QUERY 25: Team — Metrics by SVP
-- Question: "Performance by SVP" / "Which executive's book is strongest?"
-- Level: Level 1 — Standard measures correct
-- ================================================================
SELECT
  `SVP`,
  MEASURE(`GPV Percent`) AS gpv_pct,
  MEASURE(`NPV Percent`) AS npv_pct,
  MEASURE(`Total Paid`) AS total_paid,
  MEASURE(`Total GPV`) AS total_gpv,
  MEASURE(`Total NPV`) AS total_npv,
  MEASURE(`Customer Count`) AS customer_count,
  MEASURE(`Payer Count`) AS payer_count,
  MEASURE(`Unapplied Percent`) AS unapplied_pct
FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark_metrics
GROUP BY `SVP`
ORDER BY MEASURE(`Total Paid`) DESC

In [0]:
%sql
-- ================================================================
-- QUERY 26: Team — Metrics by CEL
-- Question: "Performance by engagement lead" / "CEL scorecards"
-- Level: Level 1 — Standard measures correct
-- ================================================================
SELECT
  `CEL`,
  `SVP`,
  MEASURE(`GPV Percent`) AS gpv_pct,
  MEASURE(`NPV Percent`) AS npv_pct,
  MEASURE(`Total Paid`) AS total_paid,
  MEASURE(`Total NPV`) AS total_npv,
  MEASURE(`Unapplied Percent`) AS unapplied_pct,
  MEASURE(`Customer Count`) AS customer_count
FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark_metrics
GROUP BY `CEL`, `SVP`
ORDER BY MEASURE(`Total Paid`) DESC

In [0]:
%sql
-- ================================================================
-- QUERY 27: Platform / Competitor analysis
-- Question: "Performance by claims platform" / "Where do we compete?"
-- Level: Level 1 — Standard measures correct
-- ================================================================
SELECT
  `Claims Platform`,
  `Cotiviti Edit Position`,
  `Primary 3rd Party Editor`,
  MEASURE(`GPV Percent`) AS gpv_pct,
  MEASURE(`NPV Percent`) AS npv_pct,
  MEASURE(`Total Paid`) AS total_paid,
  MEASURE(`Total GPV`) AS total_gpv,
  MEASURE(`Edits per 1000`) AS edits_per_1000,
  MEASURE(`Customer Count`) AS customer_count,
  MEASURE(`Payer Count`) AS payer_count
FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark_metrics
GROUP BY `Claims Platform`, `Cotiviti Edit Position`, `Primary 3rd Party Editor`
ORDER BY MEASURE(`Total Paid`) DESC

In [0]:
%sql
-- ================================================================
-- QUERY 28: Blues vs Non-Blues
-- Question: "Compare Blues vs non-Blues" / "BCBS performance"
-- Level: Level 1 — Standard measures correct
-- ================================================================
SELECT
  `Blues Indicator`,
  `Blues Home Host`,
  MEASURE(`GPV Percent`) AS gpv_pct,
  MEASURE(`NPV Percent`) AS npv_pct,
  MEASURE(`Total Paid`) AS total_paid,
  MEASURE(`Total GPV`) AS total_gpv,
  MEASURE(`Total NPV`) AS total_npv,
  MEASURE(`Edits per 1000`) AS edits_per_1000,
  MEASURE(`Customer Count`) AS customer_count,
  MEASURE(`Payer Count`) AS payer_count
FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark_metrics
GROUP BY `Blues Indicator`, `Blues Home Host`
ORDER BY `Blues Indicator`

In [0]:
%sql
-- ================================================================
-- QUERY 29: Disabled / Deactivated rules analysis
-- Question: "Which disabled rules had high GPV?" / "Deactivated rule impact"
-- Level: Level 1 filtered — Standard measures correct
-- ================================================================
SELECT
  `Disabled`,
  `Deactivated`,
  MEASURE(`GPV Percent`) AS gpv_pct,
  MEASURE(`NPV Percent`) AS npv_pct,
  MEASURE(`Total GPV`) AS total_gpv,
  MEASURE(`Total NPV`) AS total_npv,
  MEASURE(`Total Unapplied`) AS total_unapplied,
  MEASURE(`Unapplied Percent`) AS unapplied_pct,
  MEASURE(`DP Count`) AS dp_count,
  MEASURE(`Customer Count`) AS customer_count
FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark_metrics
GROUP BY `Disabled`, `Deactivated`
ORDER BY MEASURE(`Total Unapplied`) DESC

In [0]:
%sql
-- ================================================================
-- QUERY 30: Unapplied opportunity by Customer
-- Question: "Which customers have the most missed savings?"
-- Level: Level 1 — Derived measures
-- ================================================================
SELECT
  `Customer`,
  MEASURE(`Total Unapplied`) AS total_unapplied,
  MEASURE(`Unapplied Percent`) AS unapplied_pct,
  MEASURE(`Unapplied Opportunity Value`) AS estimated_net_opportunity,
  MEASURE(`Net Retention Rate`) AS retention_rate,
  MEASURE(`GPV Percent`) AS gpv_pct,
  MEASURE(`NPV Percent`) AS npv_pct,
  MEASURE(`Total Paid`) AS total_paid
FROM oi_dev_catalog_699543347678818.user_pooja_kabadi.ppm_benchmark_metrics
GROUP BY `Customer`
HAVING MEASURE(`Total Unapplied`) > 0
ORDER BY MEASURE(`Total Unapplied`) DESC